# Représentations vectorielles pour le TAL : du BoW aux embeddings

Le traitement automatique des langues repose sur un principe essentiel : les machines ne comprennent pas les mots, elles ne comprennent que les chiffres. Pour qu’un algorithme puisse travailler sur un texte, il faut donc commencer par le traduire en nombres — c’est ce que l’on appelle la vectorisation. Concrètement, un document devient un vecteur, une série de valeurs numériques qui le positionnent dans un espace mathématique. Et si l’on traduit plusieurs articles avec la même méthode, ils pourront être positionnés dans cet espace afin de les comparer : deux articles proches par leur contenu seront situés proches l’un de l’autre. Cette simple logique de distance suffit à automatiser des tâches complexes sans qu’aucun agent humain n’ait besoin de lire quoi que ce soit, comme classer des documents, détecter des doublons, traduire une langue, ou repérer des informations pertinentes dans un corpus.

Transformer du texte en vecteurs pose cependant un problème immédiat : les mots sont ambigus. Un même terme peut désigner des réalités très différentes selon le contexte. Prenons le mot « palais » : parle-t-on d’un monument ou de l’organe ? Les approches les plus simples, comme le *Bag-of-Words* (BoW), ne font que compter les mots sans tenir compte de leur entourage. Elles traitent chaque occurrence de « palais » de façon identique, que l’on parle de Versailles ou d’anatomie. Pour dépasser cette limite, il faut des représentations capables de saisir le contexte. C’est précisément ce qu’apportent les *word embeddings* : plutôt que d’attribuer à chaque mot un identifiant fixe, ils lui associent un vecteur dont les valeurs varient selon les contextes dans lesquels ce mot apparaît habituellement. Le sens n’est plus une propriété du mot isolé, mais le reflet de ses relations avec les mots qui l’entourent.

La vectorisation se heurte aussi au défi de la dimensionnalité. Représenter un corpus à partir de son vocabulaire revient à créer un espace à autant de dimensions qu’il y a de mots distincts. Un simple article de presse se représenterait ainsi déjà dans un espace à plusieurs centaines de dimensions quand un corpus entier le plongerait dans un espace bien plus vaste à plusieurs dizaines de milliers de dimensions ! Les vecteurs obtenus sont alors creux, remplis de zéros pour tous les mots absents d’un document, ce qui alourdit les calculs et dégrade la qualité des similarités.

## Représentations discrètes

### L’approche fréquentielle appliquée au modèle BoW

En linguistique, la représentation vectorielle d’un corpus se fait généralement via le modèle du sac de mots, ou *bag-of-words model* (BoW) en anglais. Dans ce cadre de référence, chaque document devient un vecteur dont les composantes sont les mots. On comprend ainsi rapidement que les corpus évoluent dans des espaces vectoriels de très haute dimensionnalité.

Considérons un corpus de deux textes :

```txt
(a) Le petit chat boit du lait. Le lait n’est pas bon pour les chats.
(b) Les petits chiens boivent de l’eau. L’eau irait aussi aux chats.
```

Après une phase de segmentation nettoyée par des opérations de filtrage et une autre de lemmatisation, nous obtenons le sac de mots suivant, constitutif du vocabulaire du corpus :

$$
\text{BoW} = \text{aller}, \text{aussi}, \text{boire}, \text{bon}, \text{chat}, \text{chien}, \text{eau}, \text{lait}, \text{petit}
$$

Cette liste est à comprendre désormais comme les dimensions de l’espace vectoriel du corpus. En optant pour une approche fréquentielle où à chaque composante est associée le nombre d’occurrences du mot dans le document, nous obtenons la vectorisation suivante :

$$
\begin{aligned}
\vec{A} = \begin{pmatrix}
    \text{(aller :) } &0 \\
    \text{(aussi :) } &0 \\
    \text{(boire :) } &1 \\
    \text{(bon :) } &1 \\
    \text{(chat :) } &2 \\
    \text{(chien :) } &0 \\
    \text{(eau :) } &0 \\
    \text{(lait :) } &2 \\
    \text{(petit :) } &1 \\
\end{pmatrix}
\hspace{5em}
\vec{B} = \begin{pmatrix}
    \text{(aller :) } &1 \\
    \text{(aussi :) } &1 \\
    \text{(boire :) } &1 \\
    \text{(bon :) } &0 \\
    \text{(chat :) } &1 \\
    \text{(chien :) } &1 \\
    \text{(eau :) } &2 \\
    \text{(lait :) } &0 \\
    \text{(petit :) } &1 \\
\end{pmatrix}
\end{aligned}
$$

### L’encodage 1 parmi *n*

Plus connue sous sa dénomination anglaise, *one-hot encoding*, et parfois appelée *encodage à chaud*, cette méthode consiste à représenter chaque catégorie comme un vecteur binaire où une seule position est activée (1) quand toutes les autres sont désactivées (0).

Considérons deux phrases étiquetées en parties du discours :

```txt
(a) Le/DET petit/ADJ chat/N boit/V du/DET lait/N ./PONCT
(b) Le/DET petit/ADJ chien/N boit/V de/DET l’/DET eau/N ./PONCT
```

Après la phase de pré-traitement, nous obtenons le sac de mots ci-dessous :

$$
\text{BoW} = (\text{boire}, \text{V}), (\text{chat}, \text{N}), \text{...}, (\text{petit}, \text{ADJ})
$$

On identifie toutes les catégories uniques dans deux ensembles : les lemmes et les étiquettes. En combinant ces catégories, on obtient :

$$
C = \text{boire}, \text{chat}, \text{chien}, \text{eau}, \text{lait}, \text{petit}, \text{ADJ}, \text{N}, \text{V}
$$

Chaque composante de BoW, le couple mot et étiquette, est représenté dans C par un vecteur *one-hot*, où une seule composante est égale à 1. Les vecteurs résultants sont :

$$
\begin{aligned}
\vec{BoW_1} = \begin{pmatrix}
    \text{(boire :) } &1 \\
    \text{(chat :) } &0 \\
    \text{(chien :) } &0 \\
    \text{(eau :) } &0 \\
    \text{(lait :) } &0 \\
    \text{(petit :) } &0 \\
    \text{(ADJ :) } &0 \\
    \text{(N :) } &0 \\
    \text{(V :) } &1 \\
\end{pmatrix}, \quad
\vec{BoW_2} = \begin{pmatrix}
    \text{(boire :) } &0 \\
    \text{(chat :) } &1 \\
    \text{(chien :) } &0 \\
    \text{(eau :) } &0 \\
    \text{(lait :) } &0 \\
    \text{(petit :) } &0 \\
    \text{(ADJ :) } &0 \\
    \text{(N :) } &1 \\
    \text{(V :) } &0 \\
\end{pmatrix}, \quad \text{...}, \quad
\vec{BoW_6} = \begin{pmatrix}
    \text{(boire :) } &0 \\
    \text{(chat :) } &0 \\
    \text{(chien :) } &0 \\
    \text{(eau :) } &0 \\
    \text{(lait :) } &0 \\
    \text{(petit :) } &1 \\
    \text{(ADJ :) } &1 \\
    \text{(N :) } &0 \\
    \text{(V :) } &0 \\
\end{pmatrix}
\end{aligned}
$$

### Évaluer l’importance d’un terme dans un document (TF-IDF)

Nous l’avons vu précédemment, certains termes dans un corpus sont plus importants que d’autres pour caractériser un texte par rapport à un autre, et leur importance n’est souvent pas proportionnelle à leur fréquence. De là découle la nécessité de repérer les mots vides de sens, les *stopwords*, pour les retirer du sac de mots (BoW) qui le représente. Pour autant, la méthode BoW se contente d’une mesure fréquentielle sans établir de rapport d’importance entre les termes. La matrice d’occurrences reste ainsi assez pauvre pour rendre compte de sémantique.

Une autre approche, largement répandue dans le traitement automatique du langage naturel, parvient à inférer, de l’analyse fréquentielle, une certaine valeur d’importance aux termes contenus dans le sac de mots. Cette approche repose sur deux principes : la fréquence du terme (TF) et la fréquence du terme dans le corpus (IDF) qui prêtent une signification à la rareté d’un terme.

Sans parler de robustesse, la justification de cette approche repose sur la loi de Zipf qui prévoit que la fréquence d’un terme dans un texte est liée à son rang dans l’ordre des fréquences : le mot le plus fréquent apparaîtrait dix fois plus souvent que le dixième mot le plus fréquent, cent fois plus que le centième etc. En grande partie pour cette raison, la méthode TF-IDF ne souffre pas de la présence des mots vides dans le sac de mots.

Dans sa formulation générale, l’expression mathématique vaut :

$$
\text{TF-IDF}(t, d) = \text{TF}(t, d) \cdot \ln\left(\frac{N}{\text{df}(t)}\right)
$$

Où :

- *t* est le token ;
- *d* figure le nombre total de tokens dans le document ;
- *N* représente le nombre total de documents ;
- et $\text{df}(t)$ calcule la fréquence du token dans le document.

On lui préfère toutefois souvent la version lissée afin d’éviter des divisions par zéro et de privilégier des mots très rares qui pourraient avoir une empreinte trop importante :

$$
\text{TF-IDF}(t, d) = \text{TF}(t, d) \cdot \ln\left(\frac{N}{1 + \text{df}(t)}\right)
$$

#### La fréquence du terme (TF)

De l’anglais *term frequency*, la fréquence du terme établit un rapport entre le nombre d’occurrences d’un token ($t$) dans un document et le nombre total de mots dans ce document ($d$) :

$$
\text{TF}(t, d) = \frac{t}{d}
$$

Prenons un corpus constitué de trois textes :

```txt
(A) Le petit chat boit du lait. Le lait n’est pas bon pour les chats.
(B) Les petits chiens boivent de l’eau. L’eau irait aussi aux chats.
(C) À partir d’un moment, eau ou lait, ils peuvent bien boire ce qu’ils veulent.
```

La taille en mots du document *A* est de 15, quand elle est de 13 pour le document *B* et de 16 pour le document *C*. Intéressons-nous au mot « lait » ($1$) qui apparaît deux fois dans *A* et une fois dans *C*, mais jamais dans *B*. Ses fréquences seront :

$$
\begin{aligned}
    \text{TF}_{1, A} &= \frac{2}{15} = 0,1333 \\
    \text{TF}_{1, B} &= \frac{0}{13} = 0 \\
    \text{TF}_{1, C} &= \frac{1}{16} = 0,0625 \\
\end{aligned}
$$

#### La fréquence inverse de document (IDF)

Quand la mesure TF s’attachait au terme dans un document, la mesure IDF (*inverse document frequency*) va s’intéresser à la présence du terme dans le corpus entier selon la relation suivante où $d$ représente le nombre de documents où le terme apparaît et $N$ le nombre total de documents :

$$
\text{IDF}(t) = \ln{\frac{N}{\text{df}(t)}}
$$

Dans notre exemple, la mesure IDF pour le mot « lait » vaut :

$$
\text{IDF}_1 = \ln{\frac{3}{2}} \approx 0.4055
$$

Le calcul du logarithme permet de pondérer le rapport entre $N$ et $d$ dans la mesure où, lors de l’obtention de TF, le résultat était situé dans un intervalle $[0, 1]$.

#### La mesure TF-IDF

Au final, la formule TF-IDF est un produit entre TF et IDF. Pour notre exemple avec le mot « lait » :

$$
\begin{aligned}
    \text{TF-IDF}_{1, A} &= \frac{2}{15} \cdot ln \frac{3}{2} \approx 0,0541 \\
    \text{TF-IDF}_{1, B} &= \frac{0}{13} \cdot ln \frac{3}{2} = 0 \\
    \text{TF-IDF}_{1, C} &= \frac{1}{16} \cdot ln \frac{3}{2} \approx 0,0253 \\
\end{aligned}
$$

Au regard du mot « lait », le document *A* apparaît ainsi comme plus pertinent. Notons que si nous avions opté pour le TF-IDF avec lissage, nous aurions obtenu des scores de 0.

#### Limitations du modèle

Quoique très largement utilisé pour sa facilité de mise en œuvre en dépit du coût en termes de calcul machine, la mesure TF-IDF souffre principalement de son incapacité à traiter la sémantique du terme.

Gardons par ailleurs à l’esprit que la formule accordera mécaniquement davantage d’importance aux documents très volumineux, aussi faudra-t-il penser à les pénaliser ou bien à ne travailler que sur des corpus équilibrés.

#### Normalisation des pondérations

Pour atténuer l’effet des documents de tailles différentes qui auront des valeurs plus élevées, les scores TF-IDF sont souvent normalisés. Deux normes sont retenues :

1. La norme $L^1$ qui divise chaque pondération par la somme absolue des pondérations ;
2. La norme $L^2$ qui divise chaque pondération TF-IDF par la norme euclidienne du vecteur.

La formule de la norme $L^1$ vaut :

$$
\text{Norme } L^1 : \frac{x_i}{\sum |x_i|}
$$

Si nous l’appliquons au mot « lait », nous calculons la somme absolue des pondérations :

$$
\sum |x_i| = 0.0541 + 0 + 0.0253 = 0.0794
$$

Avant de diviser chaque valeur avec :

$$
\begin{aligned}
    \text{TFIDF}_{1, A} &= \frac{0.0541}{0.0794} = 0.6814 \\
    \text{TFIDF}_{1, B} &= \frac{0}{0.0794} = 0 \\
    \text{TFIDF}_{1, C} &= \frac{0.0253}{0.0794} = 0.3186 \\
\end{aligned}
$$

Cette fois-ci, l’addition des pondérations sera toujours égale à 1.

Plus courante, la norme $L^2$ est la formule retenue par défaut par les algorithmes des bibliothèques spécialisées et s’exprime par :

$$
\text{Norme } L^2 : \frac{x_i}{\sqrt{\sum x_i^2}}
$$

Si nous calculons d’abord le dénominateur :

$$
\sqrt{\sum x_i^2} = \sqrt{(0.0541)^2 + (0)^2 + (0.0253)^2} = \sqrt{0.002927 + 0 + 0.000640} = 0.0597
$$

Nous pouvons ensuite normaliser les pondérations du mot « lait » :

$$
\begin{aligned}
    \text{TFIDF}_{1, A} &= \frac{0.0541}{0.0597} = 0.9062 \\
    \text{TFIDF}_{1, B} &= \frac{0}{0.0597} = 0 \\
    \text{TFIDF}_{1, C} &= \frac{0.0253}{0.0597} = 0.4238 \\
\end{aligned}
$$

Si avec la norme $L^2$ la somme des pondérations n’est plus égale à 1, la norme du vecteur $\vec{\text{lait}}$ l’est toutefois :

$$
\| \vec{\text{lait}} \| = \sqrt{(0.9061)^2 + (0)^2 + (0.4238)^2} = \sqrt{0.821 + 0 + 0.179} = \sqrt{1} = 1
$$

## Représentations denses

Les représentations discrètes, comme le *Bag-of-Words* ou le *One-Hot Encoding*, souffrent de deux limites majeures : elles ignorent la similarité sémantique entre les mots et génèrent des vecteurs creux de très grande dimension. Les représentations denses (ou *embeddings*) répondent à ces problèmes en projetant les mots, phrases ou documents dans un espace vectoriel continu où la proximité géométrique reflète une proximité sémantique.

Contrairement aux vecteurs creux, les *embeddings* sont des vecteurs de faible dimension où chaque composante est un nombre réel :

$$
\vec{\text{autrement}} = \begin{pmatrix}
    0.8765   \\
    1.3567   \\
    \text{…} \\
    -0.6823
\end{pmatrix}
\hspace{2em}
\vec{\text{lune}} = \begin{pmatrix}
    0.2908   \\
    -1.4002  \\
    \text{…} \\
    0.1209
\end{pmatrix}
$$

Cette densité permet de capturer des nuances de sens et des relations entre termes, tout en réduisant l’espace de calcul. Par exemple, dans un espace d’*embeddings* bien entraîné, l’opération vectorielle $\text{roi} - \text{homme} + \text{femme}$ donnera un vecteur proche de $\text{reine}$, illustrant la capacité des *embeddings* à encoder des relations sémantiques et syntaxiques.

### Word2Vec : la vectorisation de mots par Google

Word2Vec est en fait un ensemble de modèles développés par Google (Mikolov 2013) pour effectuer les tâches de vectorisation de mots. Ils prennent la forme de réseaux de neurones peu profonds, à deux couches, qui apprennent du contexte linguistique des mots afin d’en déduire une représentation vectorielle. L’intérêt de cette méthode est qu’elle suit l’hypothèse distributionnelle (Harris 1954) résumée par John Rupert Firth en 1957 par la formule : « *You shall know a word by the company it keeps.* » En d’autres termes, des mots sémantiquement proches partageront un contexte similaire, ce qui, du point de vue mathématique, se traduit par des vecteurs voisins l’un de l’autre.

Deux architectures sont proposées :
- le modèle *skip-gram* ;
- le modèle sac de mot continu (*Continuous Bag-Of-Words*, ou CBOW).

![](./figs/skip-gram-model.png)

#### Le modèle *skip-gram*

Dans le modèle skip-gram, le réseau de neurones estime, pour un mot donné, la probabilité que d’autres mots apparaissent dans son voisinage. Prenons la phrase :

> *Le petit chat boit du lait.*

Le modèle apprendra par exemple que *petit* et *boit* ont de fortes chances d’apparaître dans le contexte du mot *chat*.

L’architecture se présente sous la forme d’un vecteur d’entrée soumis à une couche cachée, puis à une couche de sortie activée par une fonction *softmax*. Cette fonction a pour effet de transformer un vecteur de valeurs réelles en un vecteur de probabilités en garantissant que la valeur de chaque composante est comprise entre 0 et 1 et que leur somme vaut 1. Elle prend l’exponentielle de chaque composante d’un vecteur et la divise par la somme de toutes ces exponentielles :

$$
\sigma(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}
$$

Lors de la phase d’entraînement, le modèle est paramétré avec une certaine fenêtre (par exemple 2). Il parcourt le corpus et apprend à partir de paires de mots. Le vecteur d’entrée, au format *one-hot* et de taille égale au vocabulaire, représente le mot en entrée. Ce vecteur est projeté dans un espace de dimension plus faible (par exemple 300), qui correspond à la représentation vectorielle du mot.

Reprenons la phrase « Le petit chat boit du lait » avec une fenêtre d’apprentissage de deux. D’abord, le modèle s’intéresse au mot « Le » et à ses deux voisins du contexte droit, « petit » et « chat ». Il les apparie pour l’entraînement : (*Le*, *petit*) et (*Le*, *chat*). Il passe ensuite au mot suivant, « petit », et cherche ses voisins dans la fenêtre spécifiée (*Le*, *chat*, *boit*) puis les apparie de nouveau : (*petit*, *Le*), (*petit*, *chat*) et (*petit*, *boit*).

Le vecteur d’entrée, de taille égale au vocabulaire, est alors composé d’une unique composante à 1 pour le mot considéré (par exemple « chat »), les autres étant nulles. La couche de sortie produit quant à elle un vecteur de probabilités, également de taille égale au vocabulaire.

#### Le modèle CBOW

Le modèle CBOW (*Continuous Bag of words*) est l’exacte symétrie du modèle *skip-gram* : pour un contexte donné, il estime quel mot du vocabulaire a le plus de chances d’apparaître au centre. Pour une fenêtre de 2, considérant le contexte « Le », « chat » et « boit », on s’attend à ce que le modèle nous propose en sortie le mot « petit ».

L’appariement est différent de celui du modèle *skip-gram*. Ici, le modèle CBOW prend en entrée les vecteurs décrivant les mots du contexte. Ces vecteurs, de dimension fixée (par exemple 300), sont généralement combinés (par exemple moyennés) pour former une représentation unique du contexte. En sortie, le modèle prédit le mot central.

Toujours pour la phrase « Le petit chat boit du lait » avec une fenêtre de 2, le modèle va d’abord apprendre de la paire ((*petit*, *chat*), *Le*), puis ((*Le*, *chat*, *boit*), *petit*), ensuite ((*Le*, *petit*, *boit*, *du*), *chat*) et ainsi de suite.

Le vecteur d’entrée est une combinaison des représentations *one-hot* des mots du contexte (par exemple une somme ou une moyenne). La couche de sortie produit enfin un vecteur de probabilités, de taille égale au vocabulaire, indiquant la probabilité que chaque mot soit le mot central.

### GloVe

Les architectures comme *skip-gram* et CBOW de Word2Vec apprennent les représentations vectorielles des mots en analysant des contextes locaux (fenêtres de mots autour d’un mot cible). Bien que cette approche soit efficace, elle ignore une information précieuse : les statistiques globales de co-occurrence dans l’ensemble du corpus. Par exemple, dans un grand corpus, les mots « glace » et « froid » co-occurrent fréquemment, tout comme « glace » et « crème », mais avec des nuances sémantiques différentes. Word2Vec ne distingue pas explicitement ces patterns globaux, car il se concentre sur des paires de mots locales.

GloVe (*Global Vectors for Word Representation*, Pennington 2014) propose une alternative en exploitant directement une matrice de co-occurrence construite à partir du corpus entier. Cette matrice compte, pour chaque paire de mots, combien de fois ils apparaissent ensemble dans une fenêtre contextuelle. L’intuition est que cette matrice encode des relations sémantiques et syntaxiques à l’échelle du corpus, ce qui permet d’apprendre des *embeddings* plus robustes, notamment pour les mots rares.

Le principe est de construire tout d’abord une matrice de co-occurrence de taille fixe afin de savoir combien de fois tous les mots du vocabulaire apparaissent dans l’entourage des autres mots. GloVe cherche alors à apprendre des vecteurs $w_i$ et $\tilde{w}_j$ (pour les mots $i$ et $j$) tels que leur produit scalaire $w_i^T \tilde{w}_j$ soit égal au logarithme du nombre de co-occurrences $X_{ij}$ :

$$
w_i^T \tilde{w}_j + b_i + \tilde{b}_j = \log(X_{ij})
$$

où $b_i$ et $\tilde{b}_j$ sont des biais spécifiques à chaque mot. L’intuition derrière est que si deux mots co-occurrent souvent ($X_{ij}$ grand), leur produit scalaire doit être élevé (vecteurs proches). Pour éviter de donner trop de poids aux co-occurrences fréquentes (ex. : mots grammaticaux comme « le » ou « de »), GloVe utilise une fonction de pondération qui réduit l’importance des paires très fréquentes.

### Limites des *word embeddings*

Les word embeddings, bien que révolutionnaires, souffrent de limites majeures qui en restreignent l’utilité pour des tâches complexes en TAL. Leur défaut le plus criant est leur caractère statique : chaque mot est représenté par un vecteur unique, quel que soit son contexte. Or, la polysémie est omniprésente en langue naturelle. Par exemple, le mot « palais » ne peut pas avoir le même vecteur dans « visiter un palais » et « ravir le palais ». Les *embeddings* traditionnels (Word2Vec, GloVe) mélangent ces sens, ce qui dégrade les performances sur des tâches nécessitant une compréhension fine, comme la traduction ou la résolution d’ambiguïtés.

Une autre limite majeure est leur incapacité à gérer les mots inconnus (OOV, *Out Of Vocabulary*). Les *embeddings* ne peuvent représenter que les mots présents dans leur vocabulaire d’entraînement. Les néologismes, les noms propres rares ou les fautes de frappe sont soit ignorés, soit associés à des vecteurs aléatoires. Bien que des approches comme FastText (basées sur des n-grams de caractères) atténuent ce problème, elles ne résolvent pas le fond : un modèle statique ne peut pas deviner le sens d’un mot jamais rencontré. Cette rigidité pose problème dans des domaines spécialisés ou pour des langues à morphologie riche, où les formes fléchies ou composées pullulent.

Enfin, les *word embeddings* ignorent la structure syntaxique et l’ordre des mots. Deux phrases comme « Le chat mange la souris » et « La souris mange le chat » auraient une représentation similaire si on se contente de moyenner les vecteurs des mots, alors que leur sens est opposé. Cette perte d’information rend ces modèles peu adaptés aux tâches nécessitant une compréhension globale, comme l’analyse de sentiments ou la réponse à des questions. Ces limites ont motivé le développement de représentations plus avancées, comme les *sentence embeddings* (pour capturer le sens des phrases) ou les modèles contextuels (comme BERT, où le vecteur d’un mot dépend de son environnement linguistique).

### Plongement de phrases

## Représentations contextuelles

## Comparaison et choix de représentation